In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

# 3. Training workflow

## 3.1 Define config file

In [3]:
from trichomecounter.utils.utils import load_config

# Get config file
cfg = load_config()

# Inspect config file
cfg

c:\Users\adria\OneDrive\Desktop\_\Programming\MachineLearning\TrichomeCounter\TrichomeCounter\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'training': {'batch_size': 8,
  'accu_steps': 4,
  'lr': 0.0001,
  'epochs': 10,
  'weight_decay': 0.0001,
  'optimizer': 'AdamW'},
 'loss': {'loss_fun': 'DensityCountLoss', 'loss_args': {'lbda_count': 0.3}},
 'target_map': {'target_map_fun': 'generate_density_map',
  'target_map_args': {'sigma': 20},
  'use_blend_maps': False},
 'model': {'model_name': 'model0_density20pretrain',
  'model_type': 'density-model',
  'activation': 'ReLUTanh',
  'pre_model_name': 'model0_density10',
  'pre_cp': 'best'},
 'transforms': {'short_side': 512,
  'brightness': 0.2,
  'imagenet_normalization': False},
 'paths': {'train_data': 'data/processed/train',
  'val_data': 'data/processed/val',
  'test_data': 'data/processed/test',
  'models': 'models',
  'outputs': 'outputs',
  'tb_logs': 'tb_logs'}}

## 3.2 Create dataloaders

In [4]:
from trichomecounter.data.data import get_dataloader

train_dataloader = get_dataloader(split="train", cfg=cfg)
val_dataloader = get_dataloader(split="val", cfg=cfg)

## 3.3 Model setup

In [6]:
from trichomecounter.utils.utils import init_model, init_loss, init_optimizer

# Init model
model, continue_training = init_model(cfg=cfg) 

# Init loss
criterion = init_loss(cfg=cfg)

# Init optimizer
optimizer = init_optimizer(model_params=model.parameters(),
                            cfg=cfg,
                            continue_training=continue_training)

# Get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Send model to device
model.to(device);

[WARNING] 'best' cp chosen. Should only be used for evaluation or transfer learning. Otherwise number of training epochs is logged incorrectly + optimizer has later state.
[INFO] Loaded model from 'C:\Users\adria\OneDrive\Desktop\_\Programming\MachineLearning\TrichomeCounter\TrichomeCounter\models\model0_density10\best_cp.pth'.


## 3.4 Training

In [ ]:
from trichomecounter.model.engine import train

train(model=model,
        cfg=cfg,
        train_dataloader=train_dataloader, 
        val_dataloader=val_dataloader, 
        optimizer=optimizer, 
        criterion=criterion, 
        device=device)